# 4. Synthetic Solar-Cycle Injection–Recovery (`select_gp_kernel` vs `v1` Control Points)

This tutorial replaces `docs/notebooks/04_Injection_Recovery_ControlPoints.ipynb`, where `ticktack 1.x` injected a high-amplitude 11-year solar oscillation into 30 annual Northern Hemisphere June–August rings (`t = 200..229`, measurement noise `1.0` permil) on `Guttler15` and optimized 15 spline control points via finite-difference `L-BFGS-B` (`sf.fit_ControlPoints()`) in **`117.0 s`** (followed by a `20 min 36 s` `emcee` run):

$$Q_{\mathrm{true}}(t) = 1.87 + 0.7 \times 1.87\sin\!\left(\frac{2\pi t}{11} + \frac{\pi}{2}\right)\;\mathrm{atoms\,cm^{-2}\,s^{-1}}$$

In `ticktack 2.0`, users can invert continuous production histories either via causal [Solin & Särkkä (2020)](https://doi.org/10.1007/s11222-019-09886-w) Hilbert-space POD (`GPProduction` + `linear_posterior` in milliseconds) or on a reflecting-boundary orthonormal DCT-II basis (`select_gp_kernel`), which precomputes the exact first-order-hold `DCTResponseOperator` once, ranks Matérn-3/2, Matérn-5/2, Squared-Exponential, and 11-year Quasi-Periodic (`QuasiPeriodicSpectrum`) priors via [MacKay (1992)](https://doi.org/10.1162/neco.1992.4.3.415) Laplace hyper-evidence in under a second, checks noise calibration via `error_scale_global`, and draws calibrated posterior trajectories via `laplace_samples`.

In [ ]:
import math, time
import jax, jax.numpy as jnp, matplotlib.pyplot as plt, numpy as np, tinygp.kernels as tk
import ticktack as tt

q0, years = 1.87, jnp.arange(200.0, 230.0)
cbm = tt.load_preset("Guttler15")
true_prod = tt.ProductionSum(
    steady=tt.SteadyProduction(q0),
    solar=tt.HarmonicSolar.from_polar(periods=11.0, amplitudes=0.7 * q0, phases=0.5 * math.pi, convention="sin"),
)
true_model = tt.RadiocarbonModel(cbm, true_prod, q_std=q0)
clean_s = tt.TreeRingSeries(years, jnp.zeros(30), jnp.ones(30), hemisphere="north", seasonal_weights="june-august", name="Synth_NH")
rng = np.random.default_rng(404)
obs_d14c = np.asarray(true_model.predict_d14c(clean_s)) + rng.normal(0.0, 1.0, size=30)
ds = tt.Dataset(tt.TreeRingSeries(years, obs_d14c, np.ones(30), hemisphere="north", seasonal_weights="june-august", name="Synth_NH"))
print(f"Synthetic injection: {ds.n_obs} June–August rings (200–229 CE), sigma=1.0 permil, peak-to-peak d14c={obs_d14c.ptp():.2f} permil")

## Causal POD GP vs Reflecting-Boundary DCT-II Kernel Discovery (`select_gp_kernel`)

While a fixed-kernel causal Solin–Särkkä `GPProduction` Wiener filter runs in sub-millisecond warm time, comparing `Matern32`, `Matern52`, `SquaredExponential`, and `QuasiPeriodic11yr` (`sho_solar`) on the padded reflecting-boundary DCT-II basis via `select_gp_kernel` automatically discovers the 11-year solar oscillation from noisy $\Delta^{14}\mathrm{C}$ rings alone via Laplace hyper-evidence $\ln\mathcal{Z}_{\mathrm{hyper}}$ and confirms unit noise calibration ($s_{\mathrm{error}} \approx 1$) without hand-tuning control-point spacing.

In [ ]:
gp_pod, cfg_pod = tt.RadiocarbonModel(cbm, tt.SteadyProduction(q0) + tt.GPProduction(tk.quasisep.Matern32(scale=3.5, sigma=0.7 * q0), n_modes=64), q_std=q0), tt.LinearMarginalConfig(offset_mode="shared", marginalize_gp=True)
_ = tt.linear_posterior(gp_pod, ds, cfg_pod)
t_p = time.perf_counter()
jax.block_until_ready(tt.linear_posterior(gp_pod, ds, cfg_pod).ring_pred_mean)
dt_pod_ms = (time.perf_counter() - t_p) * 1e3
cands = {"Matern32": "matern32", "Matern52": "matern52", "SquaredExponential": "squared_exp", "QuasiPeriodic11yr": "sho_solar"}
t0 = time.perf_counter()
sel = tt.select_gp_kernel(cbm, ds, candidates=cands, field_kind="gaussian", n_grid=128, pad_years=12.0, q_std=q0, rng_key=jax.random.PRNGKey(405))
samps = tt.laplace_samples(sel.best_result.fit, ds, n_samples=300, rng_key=jax.random.PRNGKey(406))
jax.block_until_ready(samps["q"])
dt_cold, t1 = time.perf_counter() - t0, time.perf_counter()
sel_w = tt.select_gp_kernel(cbm, ds, candidates=cands, field_kind="gaussian", n_grid=128, pad_years=12.0, q_std=q0, rng_key=jax.random.PRNGKey(407))
samps_w = tt.laplace_samples(sel_w.best_result.fit, ds, n_samples=300, rng_key=jax.random.PRNGKey(408))
jax.block_until_ready(samps_w["q"])
dt_warm = time.perf_counter() - t1
print(f"Causal POD GPProduction warm={dt_pod_ms:.2f} ms | DCT-II select_gp_kernel:\n{sel.format_table()}\nTiming vs v1 fit_ControlPoints (117.0 s): cold={dt_cold*1e3:.0f} ms ({117.0/dt_cold:.0f}x), warm={dt_warm*1e3:.0f} ms ({117.0/dt_warm:.0f}x)")

## Recovered Production History $Q(t)$ and Recovery Pulls

Overlaying the injected analytical sinusoid $Q_{\mathrm{true}}(t)$ onto `plot_production_posterior` together with a lower standardized recovery pull panel $(Q_{\mathrm{post}} - Q_{\mathrm{true}})/\sigma_Q$ confirms unbiased phase, amplitude, and uncertainty calibration across all three 11-year solar cycles.

In [ ]:
in_w = (samps["t_grid"] >= 200.0) & (samps["t_grid"] <= 230.0)
t_w, q_draws = np.asarray(samps["t_grid"][in_w]), np.asarray(samps["q"][:, in_w])
q_true = q0 + 0.7 * q0 * np.sin(2.0 * np.pi * t_w / 11.0 + 0.5 * np.pi)
fig, axes = plt.subplots(2, 1, figsize=(8.5, 5.6), sharex=True, gridspec_kw={"height_ratios": [3.0, 1.0], "hspace": 0.08})
tt.plot_production_posterior(t_w, q_draws, truth=q_true, q_std=q0, ax=axes[0], title=f"Recovered Solar Production History ({sel.best_name})")
axes[0].set_xlabel("")
q_pull = (q_draws.mean(axis=0) - q_true) / q_draws.std(axis=0)
axes[1].plot(t_w, q_pull, color="#0072B2", lw=1.6)
axes[1].axhline(0.0, color="#333333", ls="--", lw=1.0)
axes[1].fill_between(t_w, -2.0, 2.0, color="#0072B2", alpha=0.12)
axes[1].set_ylabel(r"$(Q - Q_{\mathrm{true}})/\sigma_Q$")
axes[1].set_xlabel("Calendar Year (Astronomical CE)")
axes[1].grid(True, alpha=0.25, linestyle="--")

## Synthetic $\Delta^{14}\mathrm{C}(t)$ Fit and Observation Pulls

Propagating the Wiener-filtered production trajectory back into tropospheric $\Delta^{14}\mathrm{C}(t)$ recovers the heavily damped `~2.5‰` June–August ring modulation cleanly beneath the $1.0\,\text{‰}$ measurement noise.

In [ ]:
fit_best = sel.best_result.fit
t_fine = jnp.linspace(199.5, 230.5, 250)
d14c_fine = fit_best.model.predict_fine(t_fine, hemisphere="north") + fit_best.global_offset
fig_d, axes_d = tt.plot_d14c_fit(
    ds,
    t_fine=t_fine,
    d14c_pred=d14c_fine,
    ring_pred=fit_best.d14c_pred,
    show_residuals=True,
    title="Synthetic June–August Rings: Recovered Atmospheric Fit & Pulls",
)

## One-to-One Data–Model Calibration

Checking `plot_data_model_correlation` confirms calibrated scatter around the `1:1` identity line consistent with the injected unit-permil noise ($\sigma = 1.0\,\text{‰}$).

In [ ]:
fig_corr, ax_corr = tt.plot_data_model_correlation(
    ds,
    fit_best.d14c_pred,
    title="Injection–Recovery: Observed vs Predicted Calibration",
)

Replacing finite-difference control-point L-BFGS-B (`117.0 s`) and 15-parameter random-walk `emcee` (`20 min 36 s`) with analytical DCT-II response operators, empirical-Bayes kernel selection, and exact Laplace sampling yields calibrated cosmic-ray reconstructions more than two orders of magnitude faster.